In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

In [2]:
orders = spark.read.csv("work/output/bronze/orders.csv", header=True, inferSchema=True)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/work/work/output/bronze/orders.csv.

- As Columns `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` are all timestamp values and about the delivery of order, these columns values  are reduqired and not dropping

- As Columns `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` are all timestamp values and about the delivery of order, these columns values  are reduqired and not dropping

In [58]:
orders = orders.withColumn("_is_valid", when(col("order_delivered_customer_date") <  col("order_purchase_timestamp"), False) \
           .otherwise(True))


In [59]:
orders = orders.dropDuplicates(['customer_id','order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date', 
 'order_estimated_delivery_date',
 'order_id',
 'order_purchase_timestamp',
 'order_status'])

try:
    orders_silver = spark.read.csv("work/output/silver/orders.csv", header=True, inferSchema=True)
    silver_cleaned = orders_silver.join(orders, on=['order_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(orders)
except Exception:
    final_silver = orders

final_silver.coalesce(1).write.mode("overwrite").csv("work/output/silver/orders.csv", header=True)

In [60]:
order_items =spark.read.csv("work/output/bronze/order_items.csv", header=True, inferSchema=True)

In [61]:
order_items = order_items.withColumn("_is_valid", when((col("price") < 0) | (col("freight_value") < 0), False).when(col("order_id").isNull() | col("product_id").isNull() | col("seller_id").isNull(), False).otherwise(True))
order_items = order_items.withColumn("freight_value", col("freight_value").cast('decimal')).withColumn("price", col("price").cast("decimal"))

[('freight_value', 'double'),
 ('order_id', 'string'),
 ('order_item_id', 'int'),
 ('price', 'double'),
 ('product_id', 'string'),
 ('seller_id', 'string'),
 ('shipping_limit_date', 'timestamp'),
 ('_ingested_at', 'timestamp'),
 ('_source_endpoint', 'string')]

In [62]:
order_items = order_items.dropDuplicates(['freight_value', 'order_id', 'order_item_id','price', 'product_id','seller_id','shipping_limit_date'])

try:
    order_items_silver = spark.read.csv("work/output/silver/order_items.csv", header=True, inferSchema=True)
    silver_cleaned = order_items_silver.join(order_items, on=['order_id', 'order_item_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(order_items)
except Exception:
    final_silver = order_items

final_silver.coalesce(1).write.mode("overwrite").csv("work/output/silver/order_items.csv", header=True)

+-------------+--------+-------------+-----+----------+---------+-------------------+------------+----------------+
|freight_value|order_id|order_item_id|price|product_id|seller_id|shipping_limit_date|_ingested_at|_source_endpoint|
+-------------+--------+-------------+-----+----------+---------+-------------------+------------+----------------+
|            0|       0|            0|    0|         0|        0|                  0|           0|               0|
+-------------+--------+-------------+-----+----------+---------+-------------------+------------+----------------+

None


In [66]:
customers =spark.read.csv("work/output/bronze/customers.csv", header=True, inferSchema=True)

In [67]:
customers = customers.withColumn("customer_zip_code_prefix", col("customer_zip_code_prefix").cast('string'))

[('customer_city', 'string'),
 ('customer_id', 'string'),
 ('customer_state', 'string'),
 ('customer_unique_id', 'string'),
 ('customer_zip_code_prefix', 'int'),
 ('_ingested_at', 'timestamp'),
 ('_source_endpoint', 'string')]

In [ ]:
customers = customers.withColumn("_is_valid", when(col("customer_id").isNull() | col("customer_unique_id").isNull(), False).otherwise(True))

In [71]:
customers = customers.dropDuplicates(['customer_city','customer_id','customer_state','customer_unique_id','customer_zip_code_prefix'])

try:
    customers_silver = spark.read.csv("work/output/silver/order_items.csv", header=True, inferSchema=True)
    silver_cleaned = customers_silver.join(customers, on=['customer_unique_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(customers)
except Exception:
    final_silver = customers

final_silver.coalesce(1).write.mode("overwrite").csv("work/output/silver/customers.csv", header=True)

In [72]:
payments = spark.read.csv("work/output/bronze/payments.csv", header=True, inferSchema=True)

In [75]:
payments = payments.withColumn("payment_value", col("payment_value").cast('decimal')).withColumn("_is_valid", when(col("payment_value")< 0 , False).when(col("order_id").isNull(), False).otherwise(True)) 

In [77]:
payments = payments.dropDuplicates(['order_id','payment_installments','payment_sequential','payment_type','payment_value'])

try:
    payments_silver = spark.read.csv("work/output/silver/payments.csv", header=True, inferSchema=True)
    silver_cleaned = payments_silver.join(payments, on=['order_id', 'payment_type'], how="left_anti")
    final_silver = silver_cleaned.unionByName(payments)
except Exception:
    final_silver = payments

final_silver.coalesce(1).write.mode("overwrite").csv("work/output/silver/payments.csv", header=True)

In [78]:
products = spark.read.csv("work/output/bronze/products.csv", inferSchema=True, header=True)

- As columns `product_category_name`, `product_description_lenght`, `product_name_lenght`, and `product_photos_qty` are of not required to drop these records/ cols
- Not drooping or filling these null records

In [81]:
products = products.withColumn("product_weight_g", col("product_weight_g").cast('decimal')).withColumn("product_width_cm", col("product_width_cm").cast('decimal')).withColumn("_is_valid", when(col("product_id").isNull(), False).otherwise(True))

In [84]:
products = products.dropDuplicates(['product_category_name',
 'product_description_lenght',
 'product_height_cm',
 'product_id',
 'product_length_cm',
 'product_name_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_width_cm'])

try:
    products_silver = spark.read.csv("work/output/silver/products.csv", header=True, inferSchema=True)
    silver_cleaned = products_silver.join(products, on=['product_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(products)
except Exception:
    final_silver = products

final_silver.coalesce(1).write.mode("overwrite").csv("work/output/silver/products.csv", header=True)

In [85]:
sellers = spark.read.csv("work/output/bronze/sellers.csv", header=True, inferSchema=True)

In [88]:
sellers = sellers.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast('string')).withColumn("_is_valid", when(col("seller_id").isNull(),False).otherwise(True))

In [89]:
sellers = sellers.dropDuplicates(['seller_city','seller_id','seller_state','seller_zip_code_prefix'])

In [90]:
try:
    sellers_silver = spark.read.csv("work/output/silver/sellers.csv", header=True, inferSchema=True)
    silver_cleaned = sellers_silver.join(sellers, on=['seller_id'], how="left_anti")
    final_silver = silver_cleaned.unionByName(sellers)
except Exception:
    final_silver = sellers

final_silver.coalesce(1).write.mode("overwrite").csv("work/output/silver/sellers.csv", header=True)